# Entrenamiento del moderador de contenido

**Objetivo:** entrenar un clasificador local con la data etiquetada por humanos. El cuaderno usa un baseline reproducible con TF-IDF y regresion logistica multi-etiqueta.

No usa APIs ni modelos remotos. Si luego se desea usar Transformers, deben cargarse desde una carpeta local previamente descargada.

In [ ]:
!pip3 install -q pandas scikit-learn joblib matplotlib seaborn

In [ ]:
from pathlib import Path
import json
import re

import joblib
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, f1_score, precision_score, recall_score
from sklearn.model_selection import train_test_split
from sklearn.multiclass import OneVsRestClassifier
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.pipeline import Pipeline

ROOT = Path('..').resolve()
PROCESSED_DIR = ROOT / 'datos' / 'processed'
MODELS_DIR = ROOT / 'modelos'
REPORTS_DIR = ROOT / 'resultados'

for path in [MODELS_DIR, REPORTS_DIR]:
    path.mkdir(parents=True, exist_ok=True)

DATASET_PATH = PROCESSED_DIR / 'dataset_etiquetado.jsonl'
CHUNKS_PATH = PROCESSED_DIR / 'chunks_para_etiquetar.jsonl'
print('Anotaciones esperadas:', DATASET_PATH)
print('Texto canónico:', CHUNKS_PATH)

## 1. Carga de datos etiquetados

`dataset_etiquetado.jsonl` contiene anotaciones ligeras: `chunk_id`, `labels` y metadatos, sin texto. Este cuaderno recupera siempre el `text` canónico desde `chunks_para_etiquetar.jsonl` mediante un cruce validado por `chunk_id`. También acepta archivos antiguos con `text`, pero ignora esa copia.

In [ ]:
def leer_jsonl(path):
    rows = []
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return pd.DataFrame(rows)


def normalizar_labels(value):
    if isinstance(value, list):
        labels = value
    elif isinstance(value, str) and value.strip():
        text = value.strip()
        if text.startswith('['):
            labels = json.loads(text)
        else:
            labels = re.split(r'[|,;]', text)
    else:
        labels = []
    labels = [str(x).strip() for x in labels if str(x).strip()]
    if 'limpio' in labels and len(labels) > 1:
        labels = [x for x in labels if x != 'limpio']
    return sorted(set(labels))


def cargar_dataset_etiquetado(annotations_path=DATASET_PATH, chunks_path=CHUNKS_PATH):
    """Une anotaciones ligeras con el texto canónico usando chunk_id."""
    annotations = leer_jsonl(annotations_path)
    chunks = leer_jsonl(chunks_path)

    for name, frame, required in [
        ('anotaciones', annotations, {'chunk_id', 'labels'}),
        ('chunks originales', chunks, {'chunk_id', 'text'}),
    ]:
        missing_columns = required - set(frame.columns)
        if missing_columns:
            raise ValueError(f'Faltan columnas en {name}: {sorted(missing_columns)}')

    if chunks['chunk_id'].isna().any() or chunks['chunk_id'].duplicated().any():
        raise ValueError('chunk_id debe ser único y no nulo en chunks_para_etiquetar.jsonl.')
    if annotations['chunk_id'].isna().any() or annotations['chunk_id'].duplicated().any():
        raise ValueError('dataset_etiquetado.jsonl debe tener una sola anotación consensuada por chunk_id.')

    unknown = sorted(set(annotations['chunk_id']) - set(chunks['chunk_id']))
    if unknown:
        preview = ', '.join(unknown[:5])
        raise ValueError(f'{len(unknown)} chunk_id etiquetados no existen en el dataset original: {preview}')

    # Si llega un archivo antiguo con text, se descarta esa copia y se usa la fuente canónica.
    annotations = annotations.drop(columns=['text'], errors='ignore')
    return annotations.merge(
        chunks[['chunk_id', 'text']], on='chunk_id', how='left', validate='one_to_one'
    )


if DATASET_PATH.exists():
    if not CHUNKS_PATH.exists():
        raise FileNotFoundError(f'No existe el dataset original: {CHUNKS_PATH}')
    df = cargar_dataset_etiquetado()
else:
    df = pd.DataFrame(columns=['chunk_id', 'text', 'labels'])
    print('No existe dataset etiquetado. Ejecuta el cuaderno 03 y genera el consenso ligero en datos/processed.')

if not df.empty:
    df['labels'] = df['labels'].apply(normalizar_labels)
    df['text'] = df['text'].fillna('').astype(str)
    df = df[(df['text'].str.len() >= 30) & (df['labels'].str.len() > 0)].reset_index(drop=True)

print('Filas entrenables:', len(df))
df.head()

## 2. Distribucion de etiquetas

In [ ]:
if not df.empty:
    conteo = pd.Series([label for labels in df['labels'] for label in labels]).value_counts()
    conteo.to_csv(REPORTS_DIR / 'distribucion_etiquetas.csv')
    display(conteo.to_frame('n'))
else:
    print('Sin datos para mostrar.')

## 3. Entrenamiento TF-IDF + OneVsRest

El baseline permite auditar el flujo completo antes de usar modelos mas pesados. Para datasets pequenos, `min_df=1` evita eliminar vocabulario util.

In [ ]:
if len(df) >= 10:
    mlb = MultiLabelBinarizer()
    y = mlb.fit_transform(df['labels'])
    X_train, X_test, y_train, y_test = train_test_split(
        df['text'], y, test_size=0.20, random_state=42
    )

    min_df = 1 if len(X_train) < 100 else 2
    modelo = Pipeline([
        ('tfidf', TfidfVectorizer(
            lowercase=True,
            strip_accents='unicode',
            ngram_range=(1, 2),
            min_df=min_df,
            max_features=50000
        )),
        ('clf', OneVsRestClassifier(LogisticRegression(max_iter=1000, class_weight='balanced')))
    ])

    modelo.fit(X_train, y_train)
    y_pred = modelo.predict(X_test)

    reporte = classification_report(y_test, y_pred, target_names=mlb.classes_, zero_division=0, output_dict=True)
    reporte_df = pd.DataFrame(reporte).T
    reporte_df.to_csv(REPORTS_DIR / 'metricas_clasificacion.csv')

    metricas_globales = {
        'precision_micro': float(precision_score(y_test, y_pred, average='micro', zero_division=0)),
        'recall_micro': float(recall_score(y_test, y_pred, average='micro', zero_division=0)),
        'f1_micro': float(f1_score(y_test, y_pred, average='micro', zero_division=0)),
        'f1_macro': float(f1_score(y_test, y_pred, average='macro', zero_division=0)),
        'n_train': int(len(X_train)),
        'n_test': int(len(X_test)),
        'labels': mlb.classes_.tolist(),
    }

    joblib.dump({'model': modelo, 'mlb': mlb, 'metrics': metricas_globales}, MODELS_DIR / 'moderador_tfidf_logreg.joblib')
    (REPORTS_DIR / 'metricas_globales.json').write_text(json.dumps(metricas_globales, indent=2, ensure_ascii=False), encoding='utf-8')
    display(reporte_df)
else:
    modelo = None
    mlb = None
    print('Se requieren al menos 10 chunks etiquetados para entrenar el baseline.')

## 4. Exportacion de modelo ligero para frontend

El frontend HTML de produccion puede usar un modelo liviano basado en palabras con peso por etiqueta. No reemplaza al modelo Python, pero permite una demo local sin servidor.

In [ ]:
def exportar_lexico(modelo, mlb, top_k=80):
    tfidf = modelo.named_steps['tfidf']
    clf = modelo.named_steps['clf']
    vocab = np.array(tfidf.get_feature_names_out())
    export = {'labels': mlb.classes_.tolist(), 'terms': {}}
    for label, estimator in zip(mlb.classes_, clf.estimators_):
        coef = estimator.coef_.ravel()
        idx = np.argsort(coef)[-top_k:][::-1]
        export['terms'][label] = [
            {'term': str(vocab[i]), 'weight': float(coef[i])}
            for i in idx if coef[i] > 0
        ]
    return export


if modelo is not None:
    lexico = exportar_lexico(modelo, mlb)
    salida_lexico = MODELS_DIR / 'modelo_ligero_palabras.json'
    salida_lexico.write_text(json.dumps(lexico, indent=2, ensure_ascii=False), encoding='utf-8')
    print('Modelo ligero exportado:', salida_lexico)
else:
    print('No hay modelo entrenado para exportar lexico.')

## 5. Prueba rapida de inferencia local

In [ ]:
texto_prueba = 'Ejemplo de texto para evaluar el clasificador local.'

if modelo is not None:
    pred = modelo.predict([texto_prueba])
    etiquetas = mlb.inverse_transform(pred)[0]
    print('Prediccion:', etiquetas if etiquetas else ['sin_etiqueta'])
else:
    print('Entrenar el modelo antes de probar inferencia.')